![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Building the Core RAG-Pipeline

*Project 3 — AI Product: From Model to Product*

**Team:** Zoé Doucet

**Track:** LLM

**Data:** Forest Entomology and Pathology

---
The objective is to build a Retrieval-Augmented Generation (RAG) pipeline for the Tree Health Investigator.

The pipeline will retrieve relevant textbook passages using semantic embeddings and generate evidence-based answers with PDF page citations.

The TF-IDF baseline achieved a page-level Recall@5 of 25%. Semantic retrieval will be evaluated against this baseline to determine whether it improves the retrieval of relevant evidence.

The pipeline should also handle insufficient evidence without inventing diagnoses or treatment recommendations.

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sentence_transformers
from pypdf import PdfReader
from src.functions import clean_text, chunk_document
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

from src.functions import *
from src.functions import search_chroma

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

## 0.1 Data preparation

The cleaned textbook passages from Notebook 01 are reused to build the RAG pipeline.

The same cleaning and chunking settings are maintained to ensure consistency with the TF-IDF baseline. Each chunk retains its original PDF page number for source citations.

In [3]:
# Loading the book
pdf = PdfReader("../data/raw/Forest_Entomology_and_Pathology.pdf")

# Extracting text and excluding the first 12 PDF pages
searchable_pages = [
    {
        "pdf_page": i + 1,
        "text": clean_text(page.extract_text() or "")
    }
    for i, page in enumerate(pdf.pages)
    if i >= 12]

# Verifying the number of searchable pages
print("Searchable pages:", len(searchable_pages))
print("First searchable PDF page:", searchable_pages[0]["pdf_page"])

Searchable pages: 798
First searchable PDF page: 13


In [ ]:
# Converting the list of pages into a DataFrame
pages_df = pd.DataFrame(searchable_pages)

# Using the column name expected by chunk_document()
pages_df = pages_df.rename(columns={"text": "clean_text"})

# Creating the chunks using the original settings
chunks = chunk_document(pages_df, size=150, overlap=30)

# Checking the result
print("Total chunks:", len(chunks))
print("First chunk ID:", chunks[0]["id"])
print("First chunk PDF page:", chunks[0]["pdf_page"])

Total chunks: 3274
First chunk ID: page_13_chunk_0
First chunk PDF page: 13


The chunks are ready for embedding and storage in the Chroma vector database. Maintaining the same chunks allows a consistent comparison between TF-IDF and semantic retrieval.

---
## 1. LLM track — the pipeline

Build it end to end, with the functions in `src/functions.py`:

- **Retrieval:** embed the chunks, store them (Chroma), retrieve the top `k` with metadata filters.
- **Generation:** a grounded prompt that answers only from the retrieved texts, cites their ids, and says "I don't know" otherwise.
- **Or an agent:** tools with clear descriptions (a retrieval tool, a calculator, web search), a step limit, and a trace you can read.

Run it on three or four questions from your evaluation set and read the answers in full.

#### 1.1 Embeddings and Chroma

Semantic embeddings convert textbook chunks into numerical vectors that represent their meaning.

The pretrained all-MiniLM-L6-v2 model produces 384-dimensional embeddings. Chroma will store these embeddings together with the chunk text and metadata, including PDF page numbers and chunk identifiers.

This allows the RAG pipeline to search for relevant evidence using semantic similarity.

#### Loading the embedding model

In [4]:
# Loading the embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


##### Creating or loading the Chroma database

In [5]:
# Opening the saved Chroma database
db_path = Path("../models/chroma")
db_path.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(db_path))

collection = chroma_client.get_or_create_collection(
    name="forest_textbook",
    metadata={"hnsw:space": "cosine"}
)

print("Stored chunks:", collection.count())

Stored chunks: 3274


In [7]:
# Only generats embeddings if the Chroma collection is empty
if collection.count() == 0:
    print("Creating embeddings for the first time...")

    # Collecting the text from every chunk
    texts = [chunk["text"] for chunk in chunks]

    # Converting the text into numerical embeddings
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    print("Embeddings shape:", embeddings.shape)

    # Preparing chunk IDs and PDF page metadata
    ids = [chunk["id"] for chunk in chunks]
    metadata = [
        {"pdf_page": chunk["pdf_page"]}
        for chunk in chunks
    ]

    # Storing chunks in batches
    batch_size = 500

    for start in range(0, len(chunks), batch_size):
        end = start + batch_size

        collection.upsert(
            ids=ids[start:end],
            documents=texts[start:end],
            embeddings=embeddings[start:end].tolist(),
            metadatas=metadata[start:end]
        )

    print("Embeddings saved.")

else:
    print("Existing embeddings loaded from Chroma. No regeneration needed.")

print("Stored chunks:", collection.count())

Existing embeddings loaded from Chroma. No regeneration needed.
Stored chunks: 3274


##### Generating and storing embeddings

In [8]:
# Collecting the text from every chunk
texts = [chunk["text"] for chunk in chunks]

# Converting the text into numerical embeddings
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True)

# Checking the result
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (3274, 384)


In [9]:
# Preparing chunk IDs and PDF page metadata
ids = [chunk["id"] for chunk in chunks]
metadata = [
    {"pdf_page": chunk["pdf_page"]}
    for chunk in chunks
]

# Storing chunks in batches to avoid large database requests
batch_size = 500

for start in range(0, len(chunks), batch_size):
    end = start + batch_size

    collection.upsert(
        ids=ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadata[start:end])

# Checking the number of stored chunks
print("Stored chunks:", collection.count())

Stored chunks: 3274


##### Adding printed textbook page numbers

As the page number of the book and the page number of the pdf file do not match, the page number of the printed book are added to the Chroma database.

In [10]:
# Extracting printed textbook page numbers
page_checks = []

for pdf_page in range(13, len(pdf.pages) + 1):
    page_text = pdf.pages[pdf_page - 1].extract_text() or ""

    page_checks.append({
        "pdf_page": pdf_page,
        "book_page": get_book_page(page_text)
    })

page_check_df = pd.DataFrame(page_checks)

# Keep only pages with an identified printed page number
book_pages = {
    int(row["pdf_page"]): int(row["book_page"])
    for _, row in page_check_df.dropna(subset=["book_page"]).iterrows()
}

print("Verified printed page numbers:", len(book_pages))

Verified printed page numbers: 775


In [11]:
# Adding printed book pages to existing Chroma metadata
batch_size = 500

for start in range(0, len(chunks), batch_size):
    batch = chunks[start:start + batch_size]

    batch_ids = [chunk["id"] for chunk in batch]

    batch_metadata = []

    for chunk in batch:
        pdf_page = chunk["pdf_page"]

        page_info = {"pdf_page": pdf_page}

        # Add a book page only when it was verified
        if pdf_page in book_pages:
            page_info["book_page"] = book_pages[pdf_page]

        batch_metadata.append(page_info)

    # Updating metadata without changing embeddings or documents
    collection.update(
        ids=batch_ids,
        metadatas=batch_metadata)

print("Updated metadata for", len(chunks), "chunks")
print("Stored chunks:", collection.count())

Updated metadata for 3274 chunks
Stored chunks: 3274


##### Verifying the stored data

In [12]:
# verifying the Chroma database
# Retrieving the first chunk using its ID
stored = collection.get(
    ids=[chunks[0]["id"]],
    include=["documents", "metadatas"])

# Checking the stored information
print("Chunk ID:", stored["ids"][0])
print("PDF page:", stored["metadatas"][0]["pdf_page"])
print("Text preview:", stored["documents"][0][:200])

Chunk ID: page_13_chunk_0
PDF page: 13
Text preview: Chapter 1 Introduction to and Importance of Insects Richard Redak 1.1 Introduction Insects and closely related arthropods are the dominant and most diverse forms of terrestrial and aquatic (non-marine


The 3,274 textbook chunks were converted into 384-dimensional embeddings using the pretrained all-MiniLM-L6-v2 model.

The embeddings were stored in a persistent Chroma vector database, together with the original chunk texts, unique identifiers and PDF page numbers.

#### 1.2 Semantic retrieval

Semantic retrieval identifies textbook passages that are relevant to a question based on the similarity of their embeddings.

The initial retrieval setting is k=5, which will later be compared with another value in section 2.

##### Testing retrieval with a page filter

In [13]:
# Testing retrieval restricted to PDF page 326
results = search_chroma(
    question="What signs can indicate a bark beetle attack?",
    model=model,
    collection=collection,
    k=5,
    pdf_page=326)

for i in range(len(results["ids"][0])):
    print("Chunk:", results["ids"][0][i])
    print("PDF page:", results["metadatas"][0][i]["pdf_page"])
    print("Distance:", round(results["distances"][0][i], 3))
    print()

Chunk: page_326_chunk_0
PDF page: 326
Distance: 0.485

Chunk: page_326_chunk_240
PDF page: 326
Distance: 0.612

Chunk: page_326_chunk_360
PDF page: 326
Distance: 0.619

Chunk: page_326_chunk_480
PDF page: 326
Distance: 0.807

Chunk: page_326_chunk_120
PDF page: 326
Distance: 0.835



##### Testing retrieval without a page filter

In [14]:
# Inspecting retrieved passages without a page filter
question = "What signs can indicate a bark beetle attack?"

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

for i in range(len(results["ids"][0])):
    print("PDF page:", results["metadatas"][0][i]["pdf_page"])
    print("Distance:", round(results["distances"][0][i], 3))
    print("Text:", results["documents"][0][i][:500])
    print("-" * 60)

PDF page: 771
Distance: 0.356
Text: subsequent infestations by the European spruce bark beetle. For Ecol Manage 318:167–174
------------------------------------------------------------
PDF page: 802
Distance: 0.402
Text: as stressed, ash trees can be colonized and killed by this invader (Herms and McCullough 2014). Novel associations between non-native forest insects and other non-host organisms, such as symbiotic fungi, can also lead to severe impacts, particularly by nonnative bark beetles (Wingfield et al. 2017). Many bark beetles (Scolytinae) have important associations with fungi that they introduce into trees. In some tree species, certain mutualistic fungi improve nutrient levels or other conditions for t
------------------------------------------------------------
PDF page: 596
Distance: 0.402
Text: 594 J. Sweeney et al. Fig. 17.4 Examples of signs used to survey for insects: a resin at the sites of southern pine beetle, Dendroctonus frontalis, attack (Photo credit: K. J. Dodds

##### Reviewing retrieval quality

Tests confirmed that Chroma returns chunk identifiers, PDF page numbers, cosine distances and passage texts. Metadata filtering successfully restricted results to the requested page.

Inspection of the retrieved passages showed mixed relevance. One passage described observable insect attack signs, while other results contained related but less useful information. Retrieval quality therefore requires systematic evaluation rather than relying only on similarity distances.

#### 1.3 Finding the right prompt

The generation step will use a prompt that restricts the language model to evidence retrieved from the textbook.

The prompt requires answers to cite PDF page numbers and to acknowledge when the available passages do not support a reliable answer. The assistant must not invent insect species identifications, pesticide recommendations or predictions about tree survival.

These constraints are intended to reduce unsupported claims and make answers traceable to the source material.

In [15]:
# testing the prompt function
# Retrieving textbook evidence for the question
question = "What signs can indicate a bark beetle attack?"

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

# Build and inspect the grounded prompt
prompt = build_prompt(question, results)

print(prompt[:1200])
print("\n--- END OF PROMPT ---\n")
print(prompt[-400:])


You are the polite and helpful Forest Pest Assistant, helping garden
and small forest owners understand possible insect-related tree problems.

Answer questions using ONLY the textbook sources provided below.

Rules:
- Use only information supported by the provided sources.
- Cite supporting evidence using the exact citation labels supplied with each source.
- Prefer printed textbook pages, for example [Book p. 594].
- If a source has no printed page number, use its PDF page citation.
- Never invent facts, references, or page numbers.
- Do not identify an exact insect species from insufficient symptoms.
- Do not recommend specific pesticides or dosages.
- Do not predict whether a tree will die without sufficient evidence.
- Keep answers clear, polite, and understandable for non-experts.

Clarifying questions:
- If important information is missing, ask a relevant clarifying question.
- Ask only one clarifying question at a time.
- Ask no more than five clarifying questions in total.
- 

The `build_prompt()` function was implemented in `src/functions.py`. It combines the user's question with retrieved textbook passages and instructions for evidence-based answers.

The prompt requires citations using verified printed textbook page numbers where available, with PDF page numbers as a fallback. It prohibits unsupported insect species identification, specific pesticide recommendations and unsupported predictions about tree survival.

The prompt also defines how the assistant should handle insufficient evidence and ask clarifying questions. The generated prompt was inspected to verify that the instructions, source texts and citation labels were included correctly.

The prompt is used in the complete RAG pipeline in Section 1.4, where answer generation is tested.

#### 1.4 Complete RAG Pipeline

In [16]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load the API key from the project root
load_dotenv(Path("..") / ".env")

# Create the OpenAI client
openai_client = OpenAI()

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("OpenAI client ready:", hasattr(openai_client, "responses"))

API key loaded: True
OpenAI client ready: True


In [17]:
# Generating an answer from the grounded prompt
answer = generate_answer(
    prompt=prompt,
    client=openai_client)

print(answer)

Signs can include **resin at the attack sites** and **woodpecker flaking on the tree**. These are examples associated with particular bark beetle attacks, so they do not by themselves identify the beetle species. [Book p. 594]


In [18]:
# Test the end-to-end RAG pipeline
question = "What signs can indicate a bark beetle attack?"

answer = ask_tree_assistant(
    question=question,
    model=model,
    collection=collection,
    client=openai_client,
    k=5)

print(answer)

Signs can include **resin at attack sites** and **woodpecker flaking on the bark** of an infested tree. The textbook illustrates resin at southern pine beetle attack sites and woodpecker flaking on a ponderosa pine colonized by western pine beetle. These signs alone do not identify the beetle species. [Book p. 594]


#### 1.5 Initial testing

In [19]:
# Loading the evaluation dataset from Notebook 01
eval_df = pd.read_csv("../data/evaluation.csv")

# Selecting four questions covering different situations
test_questions = eval_df[eval_df["question_id"].isin(["Q03", "Q06", "Q11", "Q13"])]

# Displaying the selected questions
display(test_questions[["question_id", "question", "answerable"]])

,question_id,question,answerable
2,Q03,What is the difference between insects that eat leaves and leaf miners?,True
5,Q06,My pine tree has small holes in its bark. Could insects be responsible?,True
10,Q11,Can drought make trees more vulnerable to insect attacks?,True
12,Q13,My spruce tree has brown needles. Which exact insect species is attacking it?,False


In [20]:
# Testing the complete RAG pipeline on four evaluation questions
for _, row in test_questions.iterrows():
    question_id = row["question_id"]
    question = row["question"]

    answer = ask_tree_assistant(
        question=question,
        model=model,
        collection=collection,
        client=openai_client,
        k=5)

    print(f"\n{'=' * 60}")
    print(f"Question {question_id}: {question}")
    print(f"Expected answerable: {row['answerable']}")
    print(f"\nAssistant answer:\n{answer}")


Question Q03: What is the difference between insects that eat leaves and leaf miners?
Expected answerable: True

Assistant answer:
Leaf-eating insects feed on the outside of leaves, consuming or removing leaf tissue; they may or may not eat the veins. Leaf miners feed as larvae between the leaf’s upper and lower surfaces, leaving characteristic patterns of damage. [PDF p. 245]

Question Q06: My pine tree has small holes in its bark. Could insects be responsible?
Expected answerable: True

Assistant answer:
Yes, insects could be responsible: some bark beetles damage trees through feeding, and pine engravers can colonize windthrown or freshly cut trees. But small holes alone aren’t enough to tell whether insects caused them or which insect may be involved. [Book p. 14] [Book p. 101]

Are there any other signs or changes to the tree, such as damaged branches or recent wind damage?

Question Q11: Can drought make trees more vulnerable to insect attacks?
Expected answerable: True

Assistan

Verifying the page number metadata for question 3.

In [21]:
# Inspecting retrieved source pages for Q03
question = test_questions.loc[
    test_questions["question_id"] == "Q03",
    "question"
].iloc[0]

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

# Displaying the citation metadata for each retrieved chunk
for i in range(len(results["ids"][0])):
    print("Chunk:", results["ids"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("-" * 40)

Chunk: page_245_chunk_120
Metadata: {'pdf_page': 245}
----------------------------------------
Chunk: page_245_chunk_0
Metadata: {'pdf_page': 245}
----------------------------------------
Chunk: page_190_chunk_240
Metadata: {'pdf_page': 190, 'book_page': 182}
----------------------------------------
Chunk: page_423_chunk_0
Metadata: {'book_page': 418, 'pdf_page': 423}
----------------------------------------
Chunk: page_190_chunk_360
Metadata: {'book_page': 182, 'pdf_page': 190}
----------------------------------------


In [22]:
# Inspecting the original header and extracted printed page
page_text = pdf.pages[244].extract_text() or ""

print("First lines of PDF page 245:")
print("\n".join(page_text.splitlines()[:5]))

print("\nDetected book page:", get_book_page(page_text))

First lines of PDF page 245:
Chapter 9 
Foliage Feeders 
Joseph Elkinton and Artemis Roehrig 
9.1 Introduction 
One´ of´ the´ most´ signiﬁcant´ categories´ of´ insects´ that´ cause´ damage´ to´ trees´ are´ the´

Detected book page: None


As no page number was detected the pdf page number is given.

**Summary**

The complete RAG pipeline was tested on four evaluation questions: three answerable questions and one requiring uncertainty.

The assistant produced relevant, textbook-grounded answers with citations. It explained leaf-feeding damage and drought-related insect vulnerability, and it avoided making definitive diagnoses from limited symptoms.

Two limitations were observed. The answer about bark holes relied on relatively general evidence, and the unanswerable species-identification question introduced a specific insect before sufficient symptoms were established. These cases highlight opportunities to improve retrieval relevance and uncertainty handling.

Citation metadata was also verified. Chapter-opening pages without a reliably extracted printed page number correctly use PDF-page citations as a fallback.

---
## 2. LLM track — one design choice, compared

Chunk size, `k`, a metadata filter, the embedding model, the prompt or the tools: change **one** thing and measure it on the evaluation set (at least the retrieval hit rate). Keep the better version, and log the cost of each run.

#### 2.1 Model and setting comparison

The semantic retriever is evaluated using the 12 answerable questions from Notebook 01. The comparison examines two embedding models, MiniLM (all-MiniLM-L6-v2) and MPNet (all-mpnet-base-v2), each tested with two retrieval settings: k=3 and k=5.

All four configurations use the same 3,274 textbook chunks and evaluation questions to ensure a consistent comparison.

A retrieval is considered successful when at least one returned chunk comes from the expected PDF page. This page-level hit rate is consistent with the TF-IDF baseline evaluation, allowing the results to be compared directly.

In addition to retrieval quality, average retrieval time and available embedding or indexing times are recorded to assess computational requirements. The results are used to select the final retrieval configuration.

#### Loading the evaluation questions

In [24]:
# Loading the evaluation questions
eval_df = pd.read_csv("../data/evaluation.csv")

# Selecting questions with known textbook evidence
answerable_df = eval_df[eval_df["answerable"] == True].copy()

# Converting page numbers to integers
answerable_df["page"] = answerable_df["page"].astype(int)

# Checking the evaluation set
print("Answerable questions:", len(answerable_df))
display(answerable_df[["question_id", "question", "page"]])

Answerable questions: 12


,question_id,question,page
0,Q01,"Are most insects harmful, or are many of them beneficial?",14
1,Q02,Do all bark beetle species cause major outbreaks in forests?,140
2,Q03,What is the difference between insects that eat leaves and leaf miners?,245
3,Q04,Can bark beetles attack hardwood trees as well as conifers?,325
4,Q05,What animals or insects naturally help control woodboring beetles?,392
5,Q06,My pine tree has small holes in its bark. Could insects be responsible?,326
6,Q07,My pine tree is losing its needles. Could insects be causing this?,65
7,Q08,Some leaves on my tree have unusual patterns inside them. Could insects be responsible?,245
8,Q09,Some leaves on my tree have missing tissue. Could insects be responsible?,245
9,Q10,Can European spruce bark beetles attack trees that have fallen during storms?,111


#### 2.1.1 Comparing settings for Mini-LM

#### Testing two different revial settings

In [25]:
comparison_results = []

# Testing two different numbers of retrieved chunks
for k in [3, 5]:

    # Testing every answerable evaluation question
    for _, row in answerable_df.iterrows():

        results = search_chroma(
            question=row["question"],
            model=model,
            collection=collection,
            k=k)

        # Collecting the PDF page numbers
        retrieved_pages = [
            item["pdf_page"]
            for item in results["metadatas"][0]]

        # Checking if the expected page was retrieved
        hit = int(row["page"] in retrieved_pages)

        # Saving the result for this question
        comparison_results.append({
            "question_id": row["question_id"],
            "k": k,
            "expected_page": row["page"],
            "retrieved_pages": retrieved_pages,
            "hit": hit})

# Converting the results into a DataFrame
comparison_df = pd.DataFrame(comparison_results)

print("Total retrieval tests:", len(comparison_df))
display(comparison_df)

Total retrieval tests: 24


,question_id,k,expected_page,retrieved_pages,hit
0,Q01,3,14,"[14, 72, 27]",1
1,Q02,3,140,"[76, 334, 78]",0
2,Q03,3,245,"[245, 245, 190]",1
3,Q04,3,325,"[802, 601, 78]",0
4,Q05,3,392,"[368, 367, 190]",0
5,Q06,3,326,"[102, 111, 790]",0
6,Q07,3,65,"[196, 513, 512]",0
7,Q08,3,245,"[432, 245, 245]",1
8,Q09,3,245,"[245, 245, 751]",1
9,Q10,3,111,"[758, 771, 601]",0


#### Comparing the retrieval settings with the baseline

In [26]:
# Calculating the retrieval hit rate for each k value
hit_rates = comparison_df.groupby("k")["hit"].mean() * 100

# Displaying the results alongside the TF-IDF baseline
print("TF-IDF baseline Recall@5: 25.0%")

for k, rate in hit_rates.items():
    print(f"Semantic retrieval Hit@{k}: {rate:.1f}%")

TF-IDF baseline Recall@5: 25.0%
Semantic retrieval Hit@3: 41.7%
Semantic retrieval Hit@5: 41.7%


Both semantic retrieval settings achieved the same page-level hit rate of 41.7%, outperforming the TF-IDF baseline Recall@5 of 25%.

#### Measuring retrieval time

In [36]:
import time

# Measuring MiniLM embedding generation without changing Chroma
texts = [chunk["text"] for chunk in chunks]

start_time = time.perf_counter()

minilm_test_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True)

minilm_embedding_time = time.perf_counter() - start_time

print(f"MiniLM embedding generation: {minilm_embedding_time:.2f} seconds")
print("Embedding shape:", minilm_test_embeddings.shape)
print("MiniLM stored chunks:", collection.count())

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM embedding generation: 76.82 seconds
Embedding shape: (3274, 384)
MiniLM stored chunks: 3274


In [27]:
import time

# Storing the retrieval times
timing_results = []

# Comparing retrieval settings
for k in [3, 5]:
    for _, row in answerable_df.iterrows():

        start_time = time.perf_counter()

        results = search_chroma(
            question=row["question"],
            model=model,
            collection=collection,
            k=k)

        # Calculating elapsed time in seconds
        elapsed_time = time.perf_counter() - start_time

        # Saving the measurement
        timing_results.append({
            "question_id": row["question_id"],
            "k": k,
            "time_seconds": elapsed_time})

# Converting measurements into a DataFrame
timing_df = pd.DataFrame(timing_results)

# Calculating average retrieval time for each setting
average_times = timing_df.groupby("k")["time_seconds"].mean()

print("Average retrieval time:")
for k, seconds in average_times.items():
    print(f"k={k}: {seconds:.3f} seconds")

Average retrieval time:
k=3: 0.013 seconds
k=5: 0.011 seconds


The average retrieval time was slightly shorter for `k=5` (0.011 seconds) than for `k=3` (0.013 seconds). However, the difference of 0.002 seconds is negligible and does not provide a strong reason to prefer either setting.

#### 2.1.2 Comparing setting for MPNet

MiniLM was selected as the initial embedding model because it is lightweight and computationally efficient. MPNet was chosen as an alternative because it is a larger pretrained model that may capture semantic relationships more effectively. Both models are compatible with Sentence Transformers, allowing a controlled comparison without changing the remaining retrieval pipeline. The evaluation will determine whether MPNet improves retrieval quality enough to justify its additional computational requirements.

MPNet embeddings are stored in a separate Chroma collection because the two embedding models produce vectors with different dimensions. 

#### Loading the embedding model

In [ ]:
# Loading the MPNet embedding model
mpnet_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Checking the embedding dimensions
print("MPNet dimensions:", mpnet_model.get_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\zoed\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zoed\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MPNet dimensions: 768


C:\Users\zoed\AppData\Local\Temp\ipykernel_36960\492348.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("MPNet dimensions:", mpnet_model.get_sentence_embedding_dimension())


In [30]:
# Creating or opening a separate collection for MPNet
mpnet_collection = chroma_client.get_or_create_collection(
    name="forest_textbook_mpnet",
    metadata={"hnsw:space": "cosine"})

# Checking whether MPNet chunks are already stored
print("MPNet stored chunks:", mpnet_collection.count())

# Confirming the original MiniLM collection is unchanged
print("MiniLM stored chunks:", collection.count())

MPNet stored chunks: 0
MiniLM stored chunks: 3274


In [ ]:
import time

# Only build the MPNet index if it is empty
if mpnet_collection.count() == 0:

    # Collecting the same textbook chunks used for MiniLM
    texts = [chunk["text"] for chunk in chunks]
    ids = [chunk["id"] for chunk in chunks]

    # Keeping the PDF and printed textbook page metadata
    metadata = []

    for chunk in chunks:
        page_info = {"pdf_page": chunk["pdf_page"]}

        if chunk["pdf_page"] in book_pages:
            page_info["book_page"] = book_pages[chunk["pdf_page"]]

        metadata.append(page_info)

    # Measure embedding generation and database storage time
    start_time = time.perf_counter()

    # Generating MPNet embeddings
    mpnet_embeddings = mpnet_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True)

    print("Embedding shape:", mpnet_embeddings.shape)

    # Saving embeddings in batches
    batch_size = 500

    for start in range(0, len(chunks), batch_size):
        end = start + batch_size

        mpnet_collection.upsert(
            ids=ids[start:end],
            documents=texts[start:end],
            embeddings=mpnet_embeddings[start:end].tolist(),
            metadatas=metadata[start:end])

    # Calculating the total indexing time
    mpnet_index_time = time.perf_counter() - start_time

    print(f"MPNet indexing time: {mpnet_index_time:.2f} seconds")

else:
    print("Existing MPNet index loaded. No new embeddings needed.")

print("MPNet stored chunks:", mpnet_collection.count())

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (3274, 768)
MPNet indexing time: 619.75 seconds
MPNet stored chunks: 3274


#### Testing different retrieval settings

In [ ]:
mpnet_results = []

# Testing the same retrieval settings as MiniLM
for k in [3, 5]:
    for _, row in answerable_df.iterrows():

        # Searching using MPNet embeddings
        results = search_chroma(
            question=row["question"],
            model=mpnet_model,
            collection=mpnet_collection,
            k=k)

        # Collecting the retrieved PDF page numbers
        retrieved_pages = [
            item["pdf_page"]
            for item in results["metadatas"][0]]

        # Checking whether the expected page was retrieved
        hit = int(row["page"] in retrieved_pages)

        # Saving the result
        mpnet_results.append({
            "question_id": row["question_id"],
            "k": k,
            "expected_page": row["page"],
            "retrieved_pages": retrieved_pages,
            "hit": hit})

# Converting the results into a DataFrame
mpnet_df = pd.DataFrame(mpnet_results)

print("Total MPNet retrieval tests:", len(mpnet_df))
display(mpnet_df)

Total MPNet retrieval tests: 24


,question_id,k,expected_page,retrieved_pages,hit
0,Q01,3,14,"[14, 13, 14]",1
1,Q02,3,140,"[107, 139, 308]",0
2,Q03,3,245,"[24, 59, 102]",0
3,Q04,3,325,"[342, 306, 122]",0
4,Q05,3,392,"[740, 368, 353]",0
5,Q06,3,326,"[104, 104, 317]",0
6,Q07,3,65,"[104, 196, 512]",0
7,Q08,3,245,"[432, 245, 245]",1
8,Q09,3,245,"[245, 789, 189]",1
9,Q10,3,111,"[771, 111, 341]",1


#### Calculating and comparing the hit rate

In [33]:
# Calculating MPNet hit rate for each k value
mpnet_hit_rates = mpnet_df.groupby("k")["hit"].mean() * 100

# Displaying the results alongside MiniLM
for k, rate in mpnet_hit_rates.items():
    print(f"MPNet Hit@{k}: {rate:.1f}%")

print("\nMiniLM Hit@3: 41.7%")
print("MiniLM Hit@5: 41.7%")
print("TF-IDF baseline Recall@5: 25.0%")

MPNet Hit@3: 41.7%
MPNet Hit@5: 50.0%

MiniLM Hit@3: 41.7%
MiniLM Hit@5: 41.7%
TF-IDF baseline Recall@5: 25.0%


MPNet achieved a page-level hit rate of 41.7% with `k=3` and 50.0% with `k=5`. The `k=5` configuration retrieved the expected evidence page for 6 of the 12 answerable questions, compared with 5 questions for both MiniLM settings.

MPNet with `k=5` therefore achieved the highest retrieval hit rate in the experiment. However, the improvement represents only one additional successful question, so the result should be interpreted cautiously given the small evaluation set.

##### Calculating and comparing the retrieval time

In [ ]:
mpnet_timing_results = []

# Testing both retrieval settings
for k in [3, 5]:
    for _, row in answerable_df.iterrows():

        # Starting measuring time
        start_time = time.perf_counter()

        # Retrieving textbook passages using MPNet
        results = search_chroma(
            question=row["question"],
            model=mpnet_model,
            collection=mpnet_collection,
            k=k)

        # Calculating elapsed time
        elapsed_time = time.perf_counter() - start_time

        # Saving the measurement
        mpnet_timing_results.append({
            "question_id": row["question_id"],
            "k": k,
            "time_seconds": elapsed_time})

# Creating a DataFrame with all timing measurements
mpnet_timing_df = pd.DataFrame(mpnet_timing_results)

# Calculating average retrieval time for each k
mpnet_average_times = (
    mpnet_timing_df.groupby("k")["time_seconds"].mean())

for k, avg_time in mpnet_average_times.items():
    print(f"MPNet k={k}: {avg_time:.3f} seconds")

MPNet k=3: 0.038 seconds
MPNet k=5: 0.036 seconds


#### 2.1.3 Comparing the models and the settings

In [38]:
# TF-IDF baseline
comparison_summary = [
    {
        "model": "TF-IDF baseline",
        "k": 5,
        "hit_rate_percent": 25.0,
        "avg_retrieval_seconds": None,
        "embedding_seconds": None,
        "indexing_seconds": None}]

# Adding both semantic embedding models
for model_name, rates, times in [
    ("MiniLM", hit_rates, average_times),
    ("MPNet", mpnet_hit_rates, mpnet_average_times)
]:
    for k in [3, 5]:
        comparison_summary.append({
            "model": model_name,
            "k": k,
            "hit_rate_percent": round(rates[k], 1),
            "avg_retrieval_seconds": round(times[k], 3),
            "embedding_seconds": (
                round(minilm_embedding_time, 2)
                if model_name == "MiniLM"
                else None
            ),
            "indexing_seconds": (
                round(mpnet_index_time, 2)
                if model_name == "MPNet"
                else None
            )})

summary_df = pd.DataFrame(comparison_summary)

display(summary_df)

,model,k,hit_rate_percent,avg_retrieval_seconds,embedding_seconds,indexing_seconds
0,TF-IDF baseline,5,25.0,NaN,NaN,NaN
1,MiniLM,3,41.7,0.013,76.82,NaN
2,MiniLM,5,41.7,0.011,76.82,NaN
3,MPNet,3,41.7,0.038,NaN,619.75
4,MPNet,5,50.0,0.036,NaN,619.75



The TF-IDF baseline achieved a page-level hit rate of 25.0%. MiniLM improved this to 41.7% with both `k=3` and `k=5`. MPNet achieved 41.7% with `k=3` and the highest hit rate of 50.0% with `k=5`.

MiniLM's embedding-generation time was measured separately without modifying its existing Chroma collection. MPNet's recorded indexing time of 619.75 seconds includes both embedding generation and database storage. These measurements are not directly equivalent.

MPNet also showed higher average retrieval latency than MiniLM, although all measured semantic retrieval times remained below 0.04 seconds. The differences between `k=3` and `k=5` within each model were negligible.

The results suggest that MPNet with `k=5` offers the best retrieval quality on this small evaluation set, at the cost of greater computational requirements.

### 2.2 The best model 

**The final retrieval configuration uses all-mpnet-base-v2 with k=5**

This configuration achieved the highest page-level retrieval hit rate of 50.0% (6 out of 12 questions), compared with 41.7% for both MiniLM settings and 25.0% for the original TF-IDF baseline.

MPNet required more computation during embedding generation and retrieval, but its average retrieval time of 0.036 seconds remains suitable for an interactive assistant. Its embeddings are already stored in Chroma, so the initial indexing process does not need to be repeated during normal use.

In [39]:
# Selecting the best retrieval settings
best_model_name = "sentence-transformers/all-mpnet-base-v2"
best_collection_name = "forest_textbook_mpnet"
best_k = 5
best_model = mpnet_model
best_collection = mpnet_collection

print("Selected embedding model:", best_model_name)
print("Selected Chroma collection:", best_collection_name)
print("Selected k:", best_k)

Selected embedding model: sentence-transformers/all-mpnet-base-v2
Selected Chroma collection: forest_textbook_mpnet
Selected k: 5


---
## 3. The final version

Which version you will evaluate in notebook 03 and use in the app, and why. Save it: the fitted pipeline with `joblib` (model track), or the index and the configuration (LLM track), in `models/`.

#### 3.1 Final configuration

The final RAG configuration uses the MPNet embedding model and retrieves the five most relevant textbook chunks from the corresponding Chroma collection. These settings were selected based on the retrieval comparison in Section 2.

The configuration also records the language model, chunking parameters, and persistent database location. Keeping these settings together supports reproducibility and allows the pipeline to be reloaded without regenerating the stored embeddings.

In [40]:
# Storing the final RAG pipeline settings
config = {
    "embedding_model": best_model_name,
    "llm_model": "gpt-6-luna",
    "chroma_path": "../models/chroma",
    "collection_name": best_collection_name,
    "k": best_k,
    "chunk_size": 150,
    "chunk_overlap": 30}

# Displaying the selected configuration
for setting, value in config.items():
    print(f"{setting}: {value}")

embedding_model: sentence-transformers/all-mpnet-base-v2
llm_model: gpt-6-luna
chroma_path: ../models/chroma
collection_name: forest_textbook_mpnet
k: 5
chunk_size: 150
chunk_overlap: 30


### 3.2 Saving the configuration

In [ ]:
import json

config_path = Path("../models/rag_config.json")

# Saving the configuration as JSON file
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=4)

print("Configuration saved to:", config_path)

Configuration saved to: ..\models\rag_config.json


In [42]:
# Check that the configuration file has been saved correctly
print("Configuration file exists:", config_path.exists())

# Reads and displays the saved JSON
with open(config_path, "r", encoding="utf-8") as file:
    saved_config = json.load(file)

display(saved_config)

Configuration file exists: True


{'embedding_model': 'sentence-transformers/all-mpnet-base-v2',
 'llm_model': 'gpt-6-luna',
 'chroma_path': '../models/chroma',
 'collection_name': 'forest_textbook_mpnet',
 'k': 5,
 'chunk_size': 150,
 'chunk_overlap': 30}

#### 3.3 Reloading and testing

In [44]:
# Reading the saved configuration
with open(config_path, "r", encoding="utf-8") as file:
    loaded_config = json.load(file)

# Reloading the selected embedding model
reloaded_model = SentenceTransformer(loaded_config["embedding_model"])

# Reopening the saved Chroma database
reloaded_chroma = chromadb.PersistentClient(path=loaded_config["chroma_path"])

# Opening the existing MPNet collection
reloaded_collection = reloaded_chroma.get_collection(name=loaded_config["collection_name"])

# Checking that the correct model and collection were loaded
print("Embedding dimensions:", reloaded_model.get_embedding_dimension())
print("Stored chunks:", reloaded_collection.count())
print("Retrieval k:", loaded_config["k"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768
Stored chunks: 3274
Retrieval k: 5


In [ ]:
# testing the retrieval

test_question = eval_df.loc[eval_df["question_id"] == "Q10", "question"].iloc[0]

# Retrieving evidence using the reloaded pipeline
reloaded_results = search_chroma(
    question=test_question,
    model=reloaded_model,
    collection=reloaded_collection,
    k=loaded_config["k"])

# Displaying the retrieved pages and their citation labels
print("Question:", test_question)
print("\nRetrieved sources:")

for metadata in reloaded_results["metadatas"][0]:
    pdf_page = metadata["pdf_page"]

    if "book_page" in metadata:
        citation = f"[Book p. {metadata['book_page']}]"
    else:
        citation = f"[PDF p. {pdf_page}]"

    print(f"PDF page {pdf_page} — {citation}")

# Verifying whether the expected evidence page was retrieved
expected_page = int(
    eval_df.loc[eval_df["question_id"] == "Q10", "page"].iloc[0])

retrieved_pages = [
    item["pdf_page"]
    for item in reloaded_results["metadatas"][0]]

print("\nExpected PDF page:", expected_page)
print("Expected page retrieved:", expected_page in retrieved_pages)

Question: Can European spruce bark beetles attack trees that have fallen during storms?

Retrieved sources:
PDF page 771 — [Book p. 770]
PDF page 111 — [Book p. 101]
PDF page 341 — [Book p. 334]
PDF page 342 — [Book p. 335]
PDF page 306 — [PDF p. 306]

Expected PDF page: 111
Expected page retrieved: True


In [46]:
# tetsing the RAG piepline
final_answer = ask_tree_assistant(
    question=test_question,
    model=reloaded_model,
    collection=reloaded_collection,
    client=openai_client,
    k=loaded_config["k"])

print("Question:", test_question)
print("\nFinal answer:")
print(final_answer)

Question: Can European spruce bark beetles attack trees that have fallen during storms?

Final answer:
Storms and blowdowns can provide temporary host material for insects to reproduce and develop, but the textbook evidence here does not specifically confirm that European spruce bark beetles attack storm-fallen trees. [Book p. 101] I don’t know based on the available textbook evidence whether this species does.


The final answer seems very cautious even if the correct page was cited. Therefore, the results will be inspected in more detail.

In [47]:
# Inspecting the retrieved text from the expected evidence page
for metadata, document in zip(
    reloaded_results["metadatas"][0],
    reloaded_results["documents"][0]):
    if metadata["pdf_page"] == 111:
        print("Source: [Book p. 101]")
        print("\nRetrieved text:")
        print(document)

Source: [Book p. 101]

Retrieved text:
(Huberty and Denno 2004; Jactel et al. 2012). 4.5.3 Extreme Weather Can Have Indirect Effects Through Trees Weather events of shorter duration than prolonged stressors like drought can also weaken trees and render them more suitable hosts for insects. For example, frosts that occur late in spring, once hardwoods have already leafed out, temporarily alter normal physiological functions, and may deplete energy reserves needed to defend against insects and diseases. Late-spring frosts have contributed to outbreaks or greater abundance of several forest insects, including twolined chestnut borer (Nichols 1968; Staley 1965), oak splendor beetle, A. biguttatus (Fabricius) (Hartmann and Blank 1992; Thomas et al. 2002), and sugar maple borer, Glycobius speciosus (Say) (Horsley et al. 2002). Ice storms, tornadoes, or other blowdowns that cause breakage of tree limbs and branches also facilitate increased survival and population growth of insects by providi

The cautious answer from the model is due to the fact that the information retrieved is more general. 

In [ ]:
# Retrieving all chunks from the expected PDF page
page_results = reloaded_collection.get(
    where={"pdf_page": 111},
    include=["documents", "metadatas"]
)

print("Chunks found on PDF page 111:", len(page_results["documents"]))

# Inspect the text of each chunk
for chunk_id, document in zip(
    page_results["ids"],
    page_results["documents"]
):
    print("\nChunk:", chunk_id)
    print(document[:700])
    print("-" * 60)

Chunks found on PDF page 111: 4

Chunk: page_111_chunk_0
4 Insect Ecology 101 Berryman 1983) (see Chapter 7). If one or both of those systems is compromised as soil moisture becomes limiting for trees, then bark beetles may be more likely to successfully colonize trees or produce more progeny or both. For example, experimental drought predisposed piñon pine, Pinus edulis, to bark beetle attack and subsequent mortality (Gaylord et al. 2013), although reduced resin flow was only partially responsible (Gaylord et al. 2015). Resistance in hardwoods has been less studied, but appears to be related to overall tree health and vigor. For instance, landscape-scale oak mortality may result from repeated drought that eventually causes imbalance in carbon sto
------------------------------------------------------------

Chunk: page_111_chunk_120
(Haavik et al. 2015). The degree to which enhanced nutritional value or reduced resistance capacity, or both, contribute to improved conditions for insect

The chunk with the correct information was retrieved as well but does not seem to have been used for the answer.

In [ ]:
# Retrieving more results to inspect the ranking
ranking_results = search_chroma(
    question=test_question,
    model=reloaded_model,
    collection=reloaded_collection,
    k=30)

# Finding the rank of the chunk containing the specific evidence
target_chunk = "page_111_chunk_360"

for rank, chunk_id in enumerate(ranking_results["ids"][0], start=1):
    if chunk_id == target_chunk:
        print("Correct evidence chunk rank:", rank)
        break
else:
    print("Correct evidence chunk not found in the top 30 results.")

Correct evidence chunk rank: 14


The correct chunk was ranked number 14 and tehrefore not used in the answer.

##### Final pipeline test and retrieval limitation

The saved MPNet pipeline was successfully reloaded with 768-dimensional embeddings, 3,274 stored textbook chunks, and `k=5`.

An end-to-end test using evaluation question Q10 retrieved the expected PDF page and generated an answer with a valid textbook citation. However, the answer abstained from confirming that European spruce bark beetles colonize storm-fallen trees.

Further inspection showed that the specific supporting passage was stored in `page_111_chunk_360`, but this chunk ranked 14th in the semantic retrieval results and was therefore excluded from the top five passages supplied to the language model.

This demonstrates a limitation of page-level retrieval evaluation: retrieving the expected page does not guarantee that the exact supporting passage reaches the language model. The cautious answer was consistent with the retrieved context, but incomplete relative to the textbook.

The selected MPNet configuration is retained for the next evaluation stage. Potential improvements include chunk-level evaluation and reranking, but these require additional testing before adoption.